# Titanic exp005 - Add Title + HasCabin

この notebook は `exp001` の真の baseline に `Title` と `HasCabin` を同時に追加して比較するためのものです。

仮説:
- `Title` に加えて `HasCabin` も足すと、単独の `Title` よりさらに精度が上がるかもしれない

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    COMP_DIR = cwd.parent
elif cwd.name == "titanic":
    COMP_DIR = cwd
else:
    COMP_DIR = Path("/home/sora/dev/kaggle/competitions/titanic")

DATA_DIR = COMP_DIR / "data"
SUBMISSION_DIR = COMP_DIR / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
OUTPUT_PATH = SUBMISSION_DIR / "exp005_title_hascabin_submission.csv"

FEATURE_COLUMNS = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "Title", "HasCabin"]
NUMERIC_COLUMNS = ["Age", "SibSp", "Parch", "Fare"]
CATEGORICAL_COLUMNS = ["Pclass", "Sex", "Embarked", "Title", "HasCabin"]

TITLE_NORMALIZATION = {
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs",
    "Lady": "Rare",
    "Countess": "Rare",
    "Dona": "Rare",
    "Dr": "Rare",
    "Rev": "Rare",
    "Col": "Rare",
    "Major": "Rare",
    "Capt": "Rare",
    "Sir": "Rare",
    "Don": "Rare",
    "Jonkheer": "Rare",
    "Master": "Master",
    "Miss": "Miss",
    "Mr": "Mr",
    "Mrs": "Mrs",
}


In [2]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    title = out["Name"].fillna("").astype(str).str.extract(r",\s*([^\.]+)\.", expand=False)
    out["Title"] = title.fillna("Rare").map(lambda x: TITLE_NORMALIZATION.get(x, "Rare")).astype(str)
    out["HasCabin"] = out["Cabin"].notna().astype(int)
    return out


def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


In [3]:
train_df = add_features(pd.read_csv(TRAIN_PATH))
test_df = add_features(pd.read_csv(TEST_PATH))

display(train_df[["Name", "Title", "HasCabin"]].head())
display(train_df.groupby("HasCabin")["Survived"].agg(["mean", "count"]))
display(train_df["Title"].value_counts())

,Name,Title,HasCabin
0,"Braund, Mr. Owen Harris",Mr,0
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs,1
2,"Heikkinen, Miss. Laina",Miss,0
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs,1
4,"Allen, Mr. William Henry",Mr,0


,mean,count
HasCabin,,
0,0.299854,687
1,0.666667,204


Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64

In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            NUMERIC_COLUMNS,
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", make_one_hot_encoder()),
                ]
            ),
            CATEGORICAL_COLUMNS,
        ),
    ],
    remainder="drop",
    sparse_threshold=0.0,
)

pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=3000,
                C=1.0,
                solver="lbfgs",
                random_state=42,
            ),
        ),
    ]
)


In [5]:
X = train_df[FEATURE_COLUMNS].copy()
y = train_df["Survived"].astype(int)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy", n_jobs=-1)

print("CV scores:", [round(score, 5) for score in scores])
print("CV mean:", round(scores.mean(), 5))
print("CV std:", round(scores.std(), 5))

CV scores: [np.float64(0.84916), np.float64(0.82022), np.float64(0.82022), np.float64(0.82022), np.float64(0.83146)]
CV mean: 0.82826
CV std: 0.01132


比較メモ:

- `exp001` baseline CV mean: `0.79687`
- `exp003` Title CV mean: `0.82828`
- `exp004` HasCabin CV mean: `0.80247`
- `exp005` Title + HasCabin CV mean: 実行結果を記入

In [6]:
pipeline.fit(X, y)
test_predictions = pipeline.predict(test_df[FEATURE_COLUMNS].copy()).astype(int)

submission_df = pd.DataFrame(
    {
        "PassengerId": test_df["PassengerId"],
        "Survived": test_predictions,
    }
)
submission_df.to_csv(OUTPUT_PATH, index=False)

print("saved:", OUTPUT_PATH)
display(submission_df.head())

saved: /home/sora/dev/kaggle/competitions/titanic/submissions/exp005_title_hascabin_submission.csv


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
